# Chunking & Embedding — Sample Pipeline

This notebook builds the **chunk → embed → index** pipeline on a manageable sample of the
cleaned complaints data, so we can iterate quickly before scaling to the full dataset.

**Steps**
1. Load a **stratified sample** (10k–15k complaints) from `data/filtered_complaints.csv`.
2. Inspect the sampling strategy and product balance.
3. Experiment with **chunk size** and **chunk overlap**.
4. Justify the final chunking choice.
5. Generate embeddings with **`all-MiniLM-L6-v2`**.
6. Build and **persist the vector store** to `vector_store/`.
7. Summarize the sampling, chunking, and embedding decisions.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the project's src package importable from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.preprocess import clean_text
from src.chunking import chunk_narrative, chunk_length_summary, count_chunks
from src.embeddings import encode_chunks, build_vector_store, DEFAULT_MODEL

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (10, 5)

# Preferred path per the task; fall back to the processed/ location if needed.
CANDIDATE_PATHS = [
    PROJECT_ROOT / "data" / "filtered_complaints.csv",
    PROJECT_ROOT / "data" / "processed" / "filtered_complaints.csv",
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), CANDIDATE_PATHS[0])

# CFPB column names
ID_COL = "Complaint ID"
PRODUCT_COL = "Product"
SUBPRODUCT_COL = "Sub-product"
ISSUE_COL = "Issue"
COMPANY_COL = "Company"
STATE_COL = "State"
DATE_COL = "Date received"
NARRATIVE_COL = "Consumer complaint narrative"

print("Embedding model:", DEFAULT_MODEL)
DATA_PATH

## 1. Load the cleaned complaints data

We load the already-cleaned, target-product-filtered dataset produced by the EDA /
preprocessing step. We only need the columns that feed the chunk text and metadata.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {len(df):,} complaints from {DATA_PATH.name}")

# Keep only rows with a usable narrative (defensive — should already be filtered)
df = df[df[NARRATIVE_COL].notna() & (df[NARRATIVE_COL].str.strip() != "")].reset_index(drop=True)
print(f"Rows with a narrative: {len(df):,}")

print("\nFull-data product distribution:")
print(df[PRODUCT_COL].value_counts())

## 2. Stratified sampling (10k–15k complaints)

**Strategy:** we draw a **stratified sample by product** so the sample preserves the
relative product proportions of the full dataset. This keeps every target product
represented (important for retrieval quality) while keeping the sample small enough to
iterate on quickly.

We target a total of **`SAMPLE_SIZE` ≈ 12,000** complaints, allocating each product a
share proportional to its frequency, and never requesting more rows than a product has.

In [ ]:
SAMPLE_SIZE = 12_000  # within the requested 10k–15k range

# Proportional allocation per product, capped at the available rows
group_sizes = df[PRODUCT_COL].value_counts()
proportions = group_sizes / group_sizes.sum()
alloc = (proportions * SAMPLE_SIZE).round().astype(int)
alloc = alloc.clip(upper=group_sizes)  # never sample more than exists

def _take(group: pd.DataFrame) -> pd.DataFrame:
    n = int(alloc.get(group.name, 0))
    return group.sample(n=min(n, len(group)), random_state=RANDOM_STATE)

sample_df = (
    df.groupby(PRODUCT_COL, group_keys=False)
    .apply(_take)
    .reset_index(drop=True)
)

print(f"Sampled {len(sample_df):,} complaints (target was {SAMPLE_SIZE:,}).")
print("\nPer-product allocation:")
print(alloc)

In [ ]:
# Compare product balance: full data vs. stratified sample
balance = pd.DataFrame(
    {
        "full_%": (df[PRODUCT_COL].value_counts(normalize=True) * 100).round(2),
        "sample_%": (sample_df[PRODUCT_COL].value_counts(normalize=True) * 100).round(2),
    }
).fillna(0)
print("Product balance (% of rows):")
print(balance)

balance.plot(kind="bar")
plt.title("Product Balance: Full Data vs. Stratified Sample")
plt.ylabel("Percentage of complaints")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 3. Experiment with chunk size and overlap

We sweep a few `(chunk_size, chunk_overlap)` combinations over a subset of narratives and
record, for each: the **total number of chunks**, **average chunks per complaint**, and
**chunk length statistics** (via `src.chunking.chunk_length_summary`).

Trade-offs:
- **Smaller chunks** → more precise retrieval but more vectors and more fragmentation of context.
- **Larger chunks** → richer context per hit but risk diluting relevance and exceeding the embedder's effective context.
- **Overlap** preserves continuity across boundaries at the cost of some redundancy.

In [ ]:
# Pre-clean narratives once; reuse across experiments
sample_df = sample_df.copy()
sample_df["clean_narrative"] = sample_df[NARRATIVE_COL].map(clean_text)

# Use a subset for fast experimentation
EXPERIMENT_N = min(2_000, len(sample_df))
experiment_texts = sample_df["clean_narrative"].head(EXPERIMENT_N).tolist()

CONFIGS = [
    (300, 30),
    (500, 50),
    (800, 100),
    (1000, 150),
]

rows = []
for size, overlap in CONFIGS:
    all_chunks = []
    for cid, text in enumerate(experiment_texts):
        all_chunks.extend(
            chunk_narrative(text, complaint_id=cid, product="", chunk_size=size, chunk_overlap=overlap)
        )
    summary = chunk_length_summary(all_chunks)
    rows.append(
        {
            "chunk_size": size,
            "overlap": overlap,
            "total_chunks": summary["num_chunks"],
            "chunks_per_complaint": round(summary["num_chunks"] / EXPERIMENT_N, 2),
            "mean_chars": summary["mean_chars"],
            "median_chars": summary["median_chars"],
            "max_chars": summary["max_chars"],
        }
    )

experiment_df = pd.DataFrame(rows)
print(f"Chunking experiment over {EXPERIMENT_N:,} narratives:\n")
experiment_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = [f"{r.chunk_size}/{r.overlap}" for r in experiment_df.itertuples()]

axes[0].bar(labels, experiment_df["total_chunks"], color="steelblue")
axes[0].set_title("Total chunks by config (size/overlap)")
axes[0].set_ylabel("Total chunks")

axes[1].bar(labels, experiment_df["mean_chars"], color="darkorange")
axes[1].set_title("Mean chunk length by config")
axes[1].set_ylabel("Mean chars per chunk")

for ax in axes:
    ax.set_xlabel("chunk_size / overlap")
plt.tight_layout()
plt.show()

## 4. Final chunking choice

**Selected: `chunk_size = 500` characters, `chunk_overlap = 50` characters (10%).**

Rationale:
- Most complaint narratives are short-to-moderate (see EDA), so a 500-char chunk usually
  captures a complete thought or paragraph without over-fragmenting.
- It keeps the **total chunk count manageable** (low storage / fast indexing) while
  splitting the long-tail narratives into a sensible number of pieces.
- A **10% overlap (50 chars)** preserves context across boundaries so a sentence split
  between two chunks is still retrievable, with minimal redundancy.
- 500 chars (~80–100 tokens) sits comfortably within `all-MiniLM-L6-v2`'s 256-token window,
  so no content is silently truncated during embedding.

> Adjust `FINAL_CHUNK_SIZE` / `FINAL_OVERLAP` below if your experiment table suggests a
> different sweet spot for your data version.

In [ ]:
FINAL_CHUNK_SIZE = 500
FINAL_OVERLAP = 50

def _get(col):
    """Return a column as a list, or a list of empty strings if it is absent."""
    return sample_df[col].tolist() if col in sample_df.columns else [""] * len(sample_df)

ids = _get(ID_COL)
products = _get(PRODUCT_COL)
subproducts = _get(SUBPRODUCT_COL)
issues = _get(ISSUE_COL)
companies = _get(COMPANY_COL)
states = _get(STATE_COL)
dates = _get(DATE_COL)
texts = sample_df["clean_narrative"].tolist()

# Build metadata-rich chunks for the full sample
chunks = []
for i in range(len(sample_df)):
    extra = {
        "product_category": products[i],
        "issue": issues[i],
        "company": companies[i],
        "state": states[i],
        "date_received": dates[i],
    }
    product = subproducts[i] if str(subproducts[i]).strip() else products[i]
    chunks.extend(
        chunk_narrative(
            text=texts[i],
            complaint_id=ids[i],
            product=product,
            chunk_size=FINAL_CHUNK_SIZE,
            chunk_overlap=FINAL_OVERLAP,
            extra_metadata=extra,
        )
    )

print(f"Built {count_chunks(chunks):,} chunks from {len(sample_df):,} complaints.")
print("Length summary:", chunk_length_summary(chunks))
print("\nExample chunk metadata:")
chunks[0].to_dict()

## 5. Generate embeddings with `all-MiniLM-L6-v2`

We embed every chunk with `sentence-transformers/all-MiniLM-L6-v2` (384-dim, normalized).
This model is a strong, lightweight default: fast on CPU, good semantic quality, and a
small footprint suitable for a local vector store.

In [ ]:
# Encode all chunks. This downloads the model on first run and may take a few minutes.
embeddings = encode_chunks(chunks, model_name=DEFAULT_MODEL, batch_size=64)
print(f"Embeddings shape: {embeddings.shape}  (n_chunks x embedding_dim)")

## 6. Build and persist the vector store

We persist the embeddings + documents + metadata into `vector_store/`. ChromaDB is the
default backend (native metadata support); pass `backend="faiss"` to use FAISS instead.

In [ ]:
PERSIST_DIR = str(PROJECT_ROOT / "vector_store")

store = build_vector_store(
    chunks,
    backend="chroma",          # or "faiss"
    persist_dir=PERSIST_DIR,
    embeddings=embeddings,     # reuse the embeddings we already computed
    collection_name="complaints",
)

print(f"Vector store persisted to: {PERSIST_DIR}")
try:
    print(f"Items in collection: {store.count():,}")
except Exception:
    print("Vector store built (FAISS index written to disk).")

## 7. Summary of decisions

**Sampling**
- Drew a **stratified-by-product** sample of **~12,000** complaints (within the 10k–15k target).
- Proportional allocation preserved the full dataset's product mix while capping at each
  product's available rows — every target product stays represented for balanced retrieval.
- Fixed `random_state=42` for reproducibility.

**Chunking**
- Swept `(300/30, 500/50, 800/100, 1000/150)` and compared total chunks, chunks-per-complaint,
  and length statistics.
- Chose **`chunk_size=500`, `overlap=50` (10%)**: complete-thought chunks, manageable chunk
  counts, boundary continuity via overlap, and comfortably within MiniLM's 256-token window.

**Embedding & vector store**
- Embedded all chunks with **`all-MiniLM-L6-v2`** (384-dim, L2-normalized) — fast, lightweight, strong quality.
- Persisted to **`vector_store/`** via **ChromaDB** (cosine space), storing per-chunk metadata:
  `complaint_id, product_category, product, issue, company, state, date_received, chunk_index, total_chunks`.
- The persisted store is now ready for the retrieval + generation pipeline (`src/rag.py`).